# MLP Backpropagation from Scratch

In [60]:
import numpy as np
import pandas as pd

## 1. Regression Problem

In [61]:
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 6]], columns=['cgpa', 'profile_score', 'lpa'])

In [62]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,6


### a. Parameter initialization

Model Architecture
- 2 inputs, 2 hidden dim, 1 output
- Activation linear
- Loss: MSE

In [63]:
def initialize_parameters(layer_dims):

    np.random.seed(3)
    parameters = {}

    L = len(layer_dims)

    for l in range(1, L):

        parameters['W' + str(l)] = np.ones((layer_dims[l], layer_dims[l-1])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

In [64]:
initialize_parameters([2, 2, 1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1, 0.1]]),
 'b2': array([[0.]])}

### b. Forward propagation

In [65]:
def linear_forward(A_prev, W, b):
    Z = (W @ A_prev) + b
    return Z

In [66]:
# Forward prop
def L_layer_forward(X, parameters):

    A = [X]
    L = len(parameters) // 2

    for l in range(1, L+1):
        a_prev = A[-1]
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]

        A.append(linear_forward(a_prev, Wl, bl))

    return A

In [67]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2, 1) # Shape(no. of features, no. of training examples)
y = df[['lpa']].values[0][0]

# Parameter initialization
parameters = initialize_parameters([2, 2, 1])

In [68]:
A = L_layer_forward(X, parameters)

print('Input, A0: ', A[0])
print('Activation from L1, A1: ', A[1])
print('Activation from L2, A2: ', A[2])

Input, A0:  [[8]
 [8]]
Activation from L1, A1:  [[1.6]
 [1.6]]
Activation from L2, A2:  [[0.32]]


### c. Backward propagation

In [69]:
def update_parameters(parameters, A, y, lr=0.001):
    y_hat = A[-1]
    A.pop()
    dL_in = 2 * (y_hat - y) # Gradient of loss wrt y_hat

    n = len(A)

    for i, a in enumerate(reversed(A)):
        Wl = parameters['W' + str(n-i)]
        bl = parameters['b' + str(n-i)]

        dL_in_prev = dL_in
        dL_in = dL_in_prev @ Wl

        dL_W = a @ dL_in_prev
        dL_B = dL_in_prev

        Wl -= lr * dL_W.T
        bl -= lr * dL_B.T

    return parameters

In [70]:
update_parameters(parameters, A, y)

{'W1': array([[0.105888, 0.105888],
        [0.105888, 0.105888]]),
 'b1': array([[0.000736],
        [0.000736]]),
 'W2': array([[0.111776, 0.111776]]),
 'b2': array([[0.00736]])}

### d. Complete Epochs

In [71]:
# Epochs implementation

parameters = initialize_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):

    loss = []

    for j in range(df.shape[0]):

        X = df[['cgpa', 'profile_score']].values[j].reshape(2, 1)
        y = df[['lpa']].values[j][0]

        # Forward propagation
        A = L_layer_forward(X, parameters)
        y_hat = A[-1]
        # Backward propagation
        update_parameters(parameters, A, y)

        loss.append((y - y_hat)**2)

    print("Epoch", i+1, 'Loss: ', np.array(loss).mean())

Epoch 1 Loss:  23.428509075497914
Epoch 2 Loss:  17.765273806116372
Epoch 3 Loss:  10.05282871164664
Epoch 4 Loss:  3.6939321270652505
Epoch 5 Loss:  1.062305048448049


In [72]:
parameters

{'W1': array([[0.26750282, 0.37150995],
        [0.26750282, 0.37150995]]),
 'b1': array([[0.02694161],
        [0.02694161]]),
 'W2': array([[0.44855407, 0.44855407]]),
 'b2': array([[0.11813432]])}

### e. Keras baseline

In [73]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Input, Dense

In [74]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2, activation='linear'),
    Dense(1, activation='linear')
])

In [75]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 2)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [76]:
new_weights = [np.array([[0.1, 0.1], [0.1, 0.1]], dtype=np.float32), np.array([0, 0], dtype=np.float32), np.array([[0.1], [0.1]], dtype=np.float32), np.array([0], dtype=np.float32)]

In [77]:
model.get_weights()

[array([[-1.0198714 ,  0.2575282 ],
        [ 0.24002707, -0.41011524]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-0.14740777],
        [-0.2661159 ]], dtype=float32),
 array([0.], dtype=float32)]

In [78]:
model.set_weights(new_weights)

In [79]:
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [80]:
optimizer = keras.optimizers.SGD(learning_rate=0.001)
model.compile(loss='mean_squared_error', optimizer=optimizer)

In [81]:
model.fit(df.iloc[:,0:-1].values, df['lpa'].values, epochs=5, verbose=1, batch_size=1)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 23.4316 
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 17.8645
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 10.2813
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 3.8779
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.1798    


In [82]:
model.get_weights()

[array([[0.2663668 , 0.2663668 ],
        [0.36872944, 0.36872944]], dtype=float32),
 array([0.02669937, 0.02669937], dtype=float32),
 array([[0.4455984],
        [0.4455984]], dtype=float32),
 array([0.1173376], dtype=float32)]

## 2. Classification Problem

In [83]:
df = pd.DataFrame([[8, 8, 1], [7, 9, 1], [6, 10, 0], [5, 5, 0]], columns=['cgpa', 'profile_score', 'placed'])

In [84]:
df.head()

,cgpa,profile_score,placed
0,8,8,1
1,7,9,1
2,6,10,0
3,5,5,0


### a. Parameter initialization

Model Architecture
- 2 inputs, 2 hidden dim, 1 output
- Activation: Sigmoid (Output neuron)
- Loss: BCE

In [90]:
parameters = initialize_parameters([2,2,1])

X = df[['cgpa', 'profile_score']].values[0].reshape(2,1) # Shape(no of features, no. of training example)
y = df[['placed']].values[0]

**Sigmoid Function**

The sigmoid function maps any real-valued input to the interval $(0, 1)$.

$$
\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

---

**Derivative of the Sigmoid Function**

Differentiate the sigmoid function with respect to $z$:

$$
\frac{d\hat{y}}{dz}
=
\frac{d}{dz}\left(\frac{1}{1+e^{-z}}\right)
$$

$$
=
-\frac{1}{(1+e^{-z})^2}
\left(-e^{-z}\right)
$$

$$
=
\frac{e^{-z}}{(1+e^{-z})^2}
$$

Using

$$
\hat{y}=\frac{1}{1+e^{-z}}
\quad\text{and}\quad
1-\hat{y}=\frac{e^{-z}}{1+e^{-z}},
$$

the derivative simplifies to

$$
\boxed{
\frac{d\hat{y}}{dz}
=
\hat{y}(1-\hat{y})
}
$$

---

**Binary Cross-Entropy (BCE) Loss**

For a single training example, the binary cross-entropy loss is

$$
L(y,\hat{y})
=
-\left(
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right)
$$

---

**Derivative of Binary Cross-Entropy Loss**

Differentiate the loss with respect to $\hat{y}$:

$$
\frac{\partial L}{\partial \hat{y}}
=
-
\left(
\frac{y}{\hat{y}}
-
\frac{1-y}{1-\hat{y}}
\right)
$$

$$
=
-\frac{y}{\hat{y}}
+
\frac{1-y}{1-\hat{y}}
$$

Combining the fractions gives

$$
\boxed{
\frac{\partial L}{\partial \hat{y}}
=
\frac{\hat{y}-y}
{\hat{y}(1-\hat{y})}
}
$$

---

**Gradient for Logistic Regression**

Since

$$
\hat{y}=\sigma(z),
$$

the chain rule gives

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial \hat{y}}
\cdot
\frac{\partial \hat{y}}{\partial z}
$$

Substituting the simplified derivatives,

$$
=
\frac{\hat{y}-y}
{\hat{y}(1-\hat{y})}
\cdot
\hat{y}(1-\hat{y})
$$

The common terms cancel, yielding

$$
\boxed{
\frac{\partial L}{\partial z}
=
\hat{y}-y
}
$$

This simplified gradient is used during gradient descent for logistic regression.

### b. Forward Propagation

In [91]:
# Utility Functions
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def linear_forward(A_prev, W, b):
    Z = (W @ A_prev) + b
    return Z

In [92]:
# Forward prop
def L_layer_forward(X, parameters):

    A = [X]
    L = len(parameters) // 2

    for l in range(1, L+1):
        a_prev = A[-1]
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]

        A.append(linear_forward(a_prev, Wl, bl))

    A[-1] = sigmoid(A[-1])

    return A

In [96]:
A = L_layer_forward(X, parameters)

print('Input, A0: ', A[0])
print('Activation from L1, A1: ', A[1])
print('Activation from L2, sigmoid(A2): ', A[2])

Input, A0:  [[8]
 [8]]
Activation from L1, A1:  [[1.6]
 [1.6]]
Activation from L2, sigmoid(A2):  [[0.57932425]]


### c. Backward propagation

In [97]:
def update_parameters(parameters, A, y, lr=0.001):
    y_hat = A[-1]
    A.pop()
    dL_in = y_hat - y # Gradient of loss wrt z (pre sigmoid activation)

    n = len(A)

    for i, a in enumerate(reversed(A)):
        Wl = parameters['W' + str(n-i)]
        bl = parameters['b' + str(n-i)]

        dL_in_prev = dL_in
        dL_in = dL_in_prev @ Wl

        dL_W = a @ dL_in_prev
        dL_B = dL_in_prev

        Wl -= lr * dL_W.T
        bl -= lr * dL_B.T

    return parameters

In [98]:
update_parameters(parameters, A, y)

{'W1': array([[0.10033654, 0.10033654],
        [0.10033654, 0.10033654]]),
 'b1': array([[4.20675748e-05],
        [4.20675748e-05]]),
 'W2': array([[0.10067308, 0.10067308]]),
 'b2': array([[0.00042068]])}

### d. Complete Epochs

In [99]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 50

for i in range(epochs):

    Loss = []

    for j in range(df.shape[0]):

        X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
        y = df[['placed']].values[j]

        # Parameter initialization
        A = L_layer_forward(X,parameters)
        y_hat = A[-1]

        update_parameters(parameters, A, y)

        Loss.append(-y*np.log(y_hat) - (1-y)*np.log(1-y_hat))

    print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

Epoch -  1 Loss -  0.6898466585481104
Epoch -  2 Loss -  0.6898045513894511
Epoch -  3 Loss -  0.6897629389031701
Epoch -  4 Loss -  0.6897218119318802
Epoch -  5 Loss -  0.6896811615148699
Epoch -  6 Loss -  0.6896409788831481
Epoch -  7 Loss -  0.6896012554546329
Epoch -  8 Loss -  0.6895619828294786
Epoch -  9 Loss -  0.6895231527855389
Epoch -  10 Loss -  0.689484757273958
Epoch -  11 Loss -  0.6894467884148889
Epoch -  12 Loss -  0.689409238493334
Epoch -  13 Loss -  0.6893720999551023
Epoch -  14 Loss -  0.6893353654028811
Epoch -  15 Loss -  0.6892990275924191
Epoch -  16 Loss -  0.6892630794288157
Epoch -  17 Loss -  0.6892275139629134
Epoch -  18 Loss -  0.6891923243877915
Epoch -  19 Loss -  0.689157504035357
Epoch -  20 Loss -  0.689123046373028
Epoch -  21 Loss -  0.6890889450005104
Epoch -  22 Loss -  0.6890551936466613
Epoch -  23 Loss -  0.6890217861664396
Epoch -  24 Loss -  0.6889887165379371
Epoch -  25 Loss -  0.6889559788594927
Epoch -  26 Loss -  0.6889235673468829

{'W1': array([[0.10075458, 0.09357433],
        [0.10075458, 0.09357433]]),
 'b1': array([[-0.00135376],
        [-0.00135376]]),
 'W2': array([[0.09465015, 0.09465015]]),
 'b2': array([[-0.01355917]])}

### e. Keras baseline

In [100]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2),
    Dense(1, activation='sigmoid')
])

In [101]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                      │ (None, 2)                   │               6 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [102]:
new_weights = [np.array([[0.1, 0.1], [0.1, 0.1]], dtype=np.float32), np.array([0, 0], dtype=np.float32), np.array([[0.1], [0.1]], dtype=np.float32), np.array([0], dtype=np.float32)]

In [103]:
model.get_weights()

[array([[ 1.1549071 ,  1.0529267 ],
        [-0.50174695, -0.6180828 ]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-1.1358106],
        [ 1.2561847]], dtype=float32),
 array([0.], dtype=float32)]

In [104]:
model.set_weights(new_weights)
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [105]:
optimizer = keras.optimizers.SGD(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [106]:
model.fit(df.iloc[:,0:-1].values, df['placed'].values, epochs=50, verbose=1, batch_size=1)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.6898 
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6898
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6898
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.6897
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.6897
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.6896
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.6896
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6896
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.6895
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.6895
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.6895
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.6894
Epoch 13/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.6894
Epoch 14/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.6893
Epoch 15/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.6893
Epoch 16/50
4/4 ━━━━━━━━━━━━━━━━━

In [107]:
model.get_weights()

[array([[0.10081298, 0.10081298],
        [0.09372851, 0.09372851]], dtype=float32),
 array([-0.00134026, -0.00134026], dtype=float32),
 array([[0.09486453],
        [0.09486453]], dtype=float32),
 array([-0.01341935], dtype=float32)]